In [107]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
from sklearn.model_selection import train_test_split
from abc import ABC
from typing import Any
from tqdm.notebook import trange
from typing import Callable

### RMSE

In [108]:
class EvaluationMetrics:
      def rmse(y_real: np.ndarray, y_predicted: np.ndarray) -> float:
        diff = (y_real - y_predicted) ** 2
        result = np.sqrt(diff.mean())
        return result

### Normalização z-score

In [109]:
class NormalizationData:
    
      def z_score(arr: np.ndarray) -> tuple[np.array, Callable[[np.ndarray], np.ndarray]]:
        mean = arr.mean()
        std = arr.std()
        out = (arr - mean) / std
        return out, lambda x: (std * x) + mean

### Gerar o modelo linear 

In [110]:
class Linear:

    w: np.ndarray
        
    def __init__(self, w: np.ndarray):
        self.w = w.reshape([-1, 1])

    def model(lenght: int, value: float) -> 'Linear':
        w = np.full(shape=lenght, fill_value=value).reshape([-1, 1])
        return Linear(w)

    def predict(self, x: np.ndarray) -> np.ndarray:
        return x @ self.w

    def update(self, w: np.ndarray):
        self.w = w.reshape([-1, 1])

    def copy_model(self):
        return Linear(self.w.reshape([-1, 1]))

In [111]:
class ParamsInicialize(ABC):
    initial_w_values: float
    seed: int
    ephocs: int
    alpha: float
    l2: float
    predictions: bool

    def __init__(self, alpha=0.01, ephocs=100, initial_w_values=1, l2=0, predictions=False, seed=1234):
        self.l2 = l2
        self.seed = seed
        self.predictions = predictions
        self.initial_w_values = initial_w_values
        self.ephocs = ephocs
        self.alpha = alpha

        def fit_train(self, x: np.ndarray, y: np.ndarray, **nargs): pass

### Algoritmo de Regressão Linear via Mínimos Quadrados Ordinários - Ordinary Least Squares (OLS)

In [112]:
class OLS(ParamsInicialize):
    
    def fit_train(self, x: np.ndarray, y: np.ndarray, **nargs) -> Linear:
        l2_matrix = np.eye(x.shape[1]) * self.l2
        w = np.linalg.inv((x.T @ x) + l2_matrix) @ x.T @ y
        return Linear(w)

### Lista 1 - Regressão linear, polinomial e regularização

In [113]:
df = pd.read_csv('data/artificial1d.csv', header=None, names=['x', 'y'])

### Normalização dos dados

In [114]:
x_normalized, _ = NormalizationData.z_score(df['x'].to_numpy())
x_normalized

array([-1.67524673, -1.55971247, -1.44417822, -1.32864396, -1.2131097 ,
       -1.09757545, -0.98204119, -0.86650693, -0.75097267, -0.63543842,
       -0.51990416, -0.4043699 , -0.28883564, -0.17330139, -0.05776713,
        0.05776713,  0.17330139,  0.28883564,  0.4043699 ,  0.51990416,
        0.63543842,  0.75097267,  0.86650693,  0.98204119,  1.09757545,
        1.2131097 ,  1.32864396,  1.44417822,  1.55971247,  1.67524673])

In [115]:
y_normalized, y_denormalized = NormalizationData.z_score(df['y'].to_numpy())
y_normalized

array([-2.05225665, -1.33142282, -1.12006895, -0.89891082, -0.33667005,
       -0.47617613, -0.82330647, -0.91048398, -0.92368912, -0.88735694,
       -0.91237789, -1.2445175 , -0.75152664, -0.49610104,  0.02986453,
        0.26860656,  0.02850739,  0.60392426,  1.18659107,  0.89184046,
        0.8931833 ,  1.03249214,  1.45552426,  0.51694346,  0.50293433,
        0.60660624,  0.72147105,  0.9917867 ,  1.29394214,  2.14064714])

In [116]:
x_ones = np.column_stack([np.ones(x_normalized.shape[0]), x_normalized])
x_ones

array([[ 1.        , -1.67524673],
       [ 1.        , -1.55971247],
       [ 1.        , -1.44417822],
       [ 1.        , -1.32864396],
       [ 1.        , -1.2131097 ],
       [ 1.        , -1.09757545],
       [ 1.        , -0.98204119],
       [ 1.        , -0.86650693],
       [ 1.        , -0.75097267],
       [ 1.        , -0.63543842],
       [ 1.        , -0.51990416],
       [ 1.        , -0.4043699 ],
       [ 1.        , -0.28883564],
       [ 1.        , -0.17330139],
       [ 1.        , -0.05776713],
       [ 1.        ,  0.05776713],
       [ 1.        ,  0.17330139],
       [ 1.        ,  0.28883564],
       [ 1.        ,  0.4043699 ],
       [ 1.        ,  0.51990416],
       [ 1.        ,  0.63543842],
       [ 1.        ,  0.75097267],
       [ 1.        ,  0.86650693],
       [ 1.        ,  0.98204119],
       [ 1.        ,  1.09757545],
       [ 1.        ,  1.2131097 ],
       [ 1.        ,  1.32864396],
       [ 1.        ,  1.44417822],
       [ 1.        ,

In [117]:
y = y_normalized.reshape([-1, 1])
y

array([[-2.05225665],
       [-1.33142282],
       [-1.12006895],
       [-0.89891082],
       [-0.33667005],
       [-0.47617613],
       [-0.82330647],
       [-0.91048398],
       [-0.92368912],
       [-0.88735694],
       [-0.91237789],
       [-1.2445175 ],
       [-0.75152664],
       [-0.49610104],
       [ 0.02986453],
       [ 0.26860656],
       [ 0.02850739],
       [ 0.60392426],
       [ 1.18659107],
       [ 0.89184046],
       [ 0.8931833 ],
       [ 1.03249214],
       [ 1.45552426],
       [ 0.51694346],
       [ 0.50293433],
       [ 0.60660624],
       [ 0.72147105],
       [ 0.9917867 ],
       [ 1.29394214],
       [ 2.14064714]])

### Questão 1

## OLS

In [119]:
ols = OLS(ephocs=100, predictions=True)

In [120]:
ols_model = ols.fit_train(x_ones, y)

In [121]:
ols_model

In [122]:
preds = ols_model.predict(x_ones)
preds

array([[-1.50352373],
       [-1.39983244],
       [-1.29614115],
       [-1.19244986],
       [-1.08875857],
       [-0.98506727],
       [-0.88137598],
       [-0.77768469],
       [-0.6739934 ],
       [-0.57030211],
       [-0.46661081],
       [-0.36291952],
       [-0.25922823],
       [-0.15553694],
       [-0.05184565],
       [ 0.05184565],
       [ 0.15553694],
       [ 0.25922823],
       [ 0.36291952],
       [ 0.46661081],
       [ 0.57030211],
       [ 0.6739934 ],
       [ 0.77768469],
       [ 0.88137598],
       [ 0.98506727],
       [ 1.08875857],
       [ 1.19244986],
       [ 1.29614115],
       [ 1.39983244],
       [ 1.50352373]])

In [123]:
x = df['x']
y = df['y']

In [124]:
fig = px.scatter(x=x, y=y, title='OLS Result')
fig.add_trace(
        go.Scatter(x=x, y=preds[:, 0])
    )
fig.show()

### Questão 2

### Regressão Polinomial

In [125]:
california_df = pd.read_csv('data/california.csv', header=None)

In [126]:
#Divisão do conjunto de dados

df_numpy = california_df.to_numpy()
x_cali = df_numpy[:, :8]
x_cali

array([[   4.1685    ,   20.        ,    6.70063694, ...,    2.37261146,
          38.8       , -121.15      ],
       [   2.8194    ,   24.        ,    4.7625    , ...,    2.53333333,
          36.75      , -119.85      ],
       [   2.425     ,   46.        ,    4.92771084, ...,    2.3253012 ,
          36.33      , -119.31      ],
       ...,
       [   6.4755    ,   36.        ,    5.81294964, ...,    2.69304556,
          34.01      , -118.45      ],
       [   3.5353    ,   34.        ,    6.31415929, ...,    2.67920354,
          37.97      , -122.3       ],
       [   1.2545    ,   41.        ,    4.30601093, ...,    4.28142077,
          36.75      , -119.76      ]])

In [127]:
def reshape_array(x: np.ndarray) -> np.ndarray: 
    return x.reshape([-1, 1])

In [128]:
y_cali = reshape_array(df_numpy[:, 8])
y_cali

array([[2.175],
       [0.811],
       [0.849],
       ...,
       [4.058],
       [1.647],
       [0.455]])

In [129]:
x_cali_norm = np.empty_like(x_cali)

for i, col in enumerate(x_cali.T):
    norm, _= NormalizationData.z_score(col)
    x_cali_norm[:, i] = norm

In [130]:
y_cali_norm, y_cali_denormalized = NormalizationData.z_score(y_cali)

In [131]:
class Poly:

    def polynomial(x: np.ndarray, order: int, bias=False) -> np.ndarray:

        if bias: x = np.column_stack([np.ones(x.shape[0]), x])

        poly = [
            x[:, p] ** i for p in range(1, x.shape[1]) for i in range(2, order)
        ]

        return np.column_stack([x, *poly])

    def tranning_polinomial(order, l2=0):
    
        order_values = [i for i in range(1, order)]
    
        ols = OLS(l2=l2)
    
        rmse_train = []
        rmse_test = []
    
        x_train,x_test,y_train,y_test=train_test_split(x_cali_norm,y_cali_norm, test_size=0.2, random_state=20)
    
        for order in order_values:
            x_train_poly = polynomial(x_train, order, bias=True)
            x_test_poly = polynomial(x_test, order, bias=True)
        
            model= ols.fit_train(x_train_poly, y_train)
        
            predited_train = model.predict(x_train_poly)
            predited_test = model.predict(x_test_poly)
        
            rmse_train.append(EvaluationMetrics.rmse(y_cali_denormalized(y_train), y_cali_denormalized(predited_train)))
            rmse_test.append(EvaluationMetrics.rmse(y_cali_denormalized(y_test), y_cali_denormalized(predited_test)))
        
        plots = [
            go.Scatter(x=order_values, y=rmse_train, name='Train RMSE'),
            go.Scatter(x=order_values, y=rmse_test, name='Test RMSE')
        ]
        
        fig = go.Figure(data=plots)
        return fig

In [132]:
result = Poly.tranning_polinomial(11)
result

In [133]:
result = Poly.tranning_polinomial(11, l2=0.001)
result